# AI Startup Security Pipeline Setup — Module 1

This notebook:

- Runs **Bandit**, **Semgrep**, **PyLint**, and **Safety** on the ML training codebase
- Analyzes vulnerabilities by severity and CWE
- Generates charts for severity and CWE distribution
- Produces all three assignment deliverables:
  1. **Security Scan Results Report** (`deliverables/Module1_Security_Report.md`)
  2. **GitHub Actions Workflow** (`deliverables/security-scan.yml`)
  3. **Remediation Priority Matrix** (`deliverables/remediation_matrix.csv`)

Run this notebook from the `module1_Intro/` directory.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("security-reports", exist_ok=True)
os.makedirs("deliverables", exist_ok=True)

print("Environment ready. Folders created: security-reports/, deliverables/")

## 1. Install tools (if needed)

In [ ]:
!pip install bandit semgrep pylint safety -q
print("Tools installed.")

## 2. Run security scans

This cell runs:
- Bandit (Python security linter)
- Semgrep (pattern-based scanner)
- PyLint (warnings/errors only)
- Safety (dependency vulnerabilities)

Outputs are written to `security-reports/`.

In [ ]:
# Bandit
!bandit -r . -f json -o security-reports/bandit-results.json || true

# Semgrep
!semgrep --config=auto . --json --output security-reports/semgrep-results.json || true

# PyLint (warnings + errors only)
!pylint *.py --disable=all --enable=W,E > security-reports/pylint-results.txt || true

# Safety
!safety check --json > security-reports/safety-results.json || true

print("All scans executed. Reports saved in security-reports/.")

## 3. Load scan results into Python

In [ ]:
with open("security-reports/bandit-results.json") as f:
    bandit = json.load(f)

with open("security-reports/semgrep-results.json") as f:
    semgrep = json.load(f)

with open("security-reports/safety-results.json") as f:
    safety = json.load(f)

print("Loaded Bandit, Semgrep, Safety results.")

## 4. Bandit findings — dataframe and summaries

In [ ]:
bandit_findings = [
    {
        "severity": r.get("issue_severity"),
        "confidence": r.get("issue_confidence"),
        "test_id": r.get("test_id"),
        "cwe": r.get("issue_cwe", {}).get("id"),
        "file": r.get("filename"),
        "line": r.get("line_number"),
        "text": r.get("issue_text"),
    }
    for r in bandit.get("results", [])
]

df_bandit = pd.DataFrame(bandit_findings)
df_bandit.head()

In [ ]:
print("=== Bandit Severity Breakdown ===")
if not df_bandit.empty:
    print(df_bandit["severity"].value_counts(), "\n")
else:
    print("No Bandit findings.\n")

print("=== Bandit Findings by CWE ===")
if not df_bandit.empty:
    print(df_bandit.groupby("cwe").size(), "\n")
else:
    print("No Bandit findings.\n")

print("=== Bandit Findings by File ===")
if not df_bandit.empty:
    print(df_bandit.groupby("file").size(), "\n")
else:
    print("No Bandit findings.\n")

## 5. Semgrep findings — dataframe and summaries

In [ ]:
semgrep_findings = [
    {
        "rule": r.get("check_id"),
        "file": r.get("path"),
        "line": r.get("start", {}).get("line"),
        "message": r.get("extra", {}).get("message"),
        "cwe": r.get("extra", {}).get("metadata", {}).get("cwe", None),
    }
    for r in semgrep.get("results", [])
]

df_semgrep = pd.DataFrame(semgrep_findings)
df_semgrep.head()

In [ ]:
print("=== Semgrep Findings by Rule ===")
if not df_semgrep.empty:
    print(df_semgrep.groupby("rule").size(), "\n")
else:
    print("No Semgrep findings.\n")

print("=== Semgrep Findings by File ===")
if not df_semgrep.empty:
    print(df_semgrep.groupby("file").size(), "\n")
else:
    print("No Semgrep findings.\n")

## 6. Safety and PyLint outputs (raw view)

In [ ]:
print("=== Safety Vulnerabilities (raw JSON) ===")
print(safety)

In [ ]:
print("=== PyLint Warnings/Errors ===")
pylint_path = "security-reports/pylint-results.txt"
if os.path.exists(pylint_path):
    with open(pylint_path) as f:
        print(f.read())
else:
    print("No PyLint output file found.")

## 7. Charts — Bandit severity and CWE distribution

In [ ]:
# Bandit severity bar chart
if not df_bandit.empty:
    severity_counts = df_bandit["severity"].value_counts()

    plt.figure(figsize=(6, 4))
    severity_counts.plot(kind="bar", color=["red", "orange", "yellow", "green"])
    plt.title("Bandit Severity Breakdown")
    plt.xlabel("Severity")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No Bandit findings to plot.")

In [ ]:
# CWE bar chart (top 10)
if not df_bandit.empty:
    cwe_counts = df_bandit["cwe"].value_counts().head(10)

    plt.figure(figsize=(8, 4))
    cwe_counts.plot(kind="bar")
    plt.title("Top CWE Categories (Bandit)")
    plt.xlabel("CWE ID")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No Bandit findings to plot CWEs.")

## 8. Build Top 10 Critical Vulnerabilities (risk scoring)

We approximate risk as:

- Map severity → numeric score
- Assume a default exploitability score
- Risk score = severity_score × exploitability

In [ ]:
severity_score = {"LOW": 1, "MEDIUM": 2, "HIGH": 3, "CRITICAL": 4}

if not df_bandit.empty:
    df_bandit["severity_score"] = df_bandit["severity"].map(severity_score).fillna(1)
    df_bandit["exploitability"] = 2  # simple default
    df_bandit["risk_score"] = df_bandit["severity_score"] * df_bandit["exploitability"]

    top10 = df_bandit.sort_values("risk_score", ascending=False).head(10)
    top10_display = top10[["file", "line", "test_id", "cwe", "severity", "risk_score", "text"]]
    top10_display
else:
    top10 = pd.DataFrame()
    print("No Bandit findings; cannot build Top 10 table.")

## 9. Generate Deliverable 1 — Security Scan Results Report (Markdown)

Creates: `deliverables/Module1_Security_Report.md`

In [ ]:
report_path = "deliverables/Module1_Security_Report.md"

with open(report_path, "w") as f:
    f.write("# Security Scan Results Report\n\n")
    f.write("## Executive Summary\n")
    f.write("This report summarizes the results of automated static analysis using Bandit, Semgrep, PyLint, and Safety on the ML training codebase. ")
    f.write("The goal is to identify hardcoded credentials, unsafe pickle operations, and exposed API keys, then prioritize remediation.\n\n")

    f.write("## Tool Summary\n")
    f.write(f"- Bandit findings: {len(df_bandit)}\\n")
    f.write(f"- Semgrep findings: {len(df_semgrep)}\\n")
    f.write(f"- Safety vulnerabilities (raw entries): {len(safety)}\\n\\n")

    f.write("## Bandit Severity Breakdown\n\n")
    if not df_bandit.empty:
        f.write(df_bandit["severity"].value_counts().to_markdown() + "\n\n")
    else:
        f.write("_No Bandit findings._\n\n")

    f.write("## Bandit Findings by CWE\n\n")
    if not df_bandit.empty:
        f.write(df_bandit.groupby("cwe").size().to_markdown() + "\n\n")
    else:
        f.write("_No Bandit findings._\n\n")

    f.write("## Semgrep Findings by Rule\n\n")
    if not df_semgrep.empty:
        f.write(df_semgrep.groupby("rule").size().to_markdown() + "\n\n")
    else:
        f.write("_No Semgrep findings._\n\n")

    f.write("## Top 10 Critical Vulnerabilities (Bandit-based)\n\n")
    if not top10.empty:
        f.write(top10[["file", "line", "test_id", "cwe", "severity", "risk_score", "text"]].to_markdown() + "\n\n")
    else:
        f.write("_No critical vulnerabilities identified._\n\n")

    f.write("## Remediation Recommendations (High Level)\n")
    f.write("- Remove hardcoded credentials and move them to environment variables or secret managers.\\n")
    f.write("- Replace unsafe pickle deserialization with safer formats (e.g., JSON, ONNX, safetensors).\\n")
    f.write("- Add input validation and strict allowlists for file paths and external inputs.\\n")
    f.write("- Enforce automated scanning in CI/CD using GitHub Actions.\\n")

print(f"Deliverable 1 written to {report_path}")

## 10. Generate Deliverable 2 — GitHub Actions Workflow (YAML)

Creates: `deliverables/security-scan.yml`

In [ ]:
workflow_path = "deliverables/security-scan.yml"

workflow = """name: Security Scan

on:
  pull_request:
    branches: [ main, master ]

jobs:
  security-scan:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install dependencies
        run: |
          pip install bandit semgrep safety

      - name: Run Bandit
        run: |
          bandit -r . -f json -o security-reports/bandit-results.json || true

      - name: Run Semgrep
        run: |
          semgrep --config=auto . --json --output security-reports/semgrep-results.json || true

      - name: Run Safety
        run: |
          safety check --json > security-reports/safety-results.json || true

      - name: Upload security reports
        uses: actions/upload-artifact@v4
        with:
          name: security-reports
          path: security-reports/

      - name: Fail on HIGH severity (Bandit)
        run: |
          python - << 'EOF'
          import json
          import sys
          try:
              with open('security-reports/bandit-results.json') as f:
                  data = json.load(f)
          except FileNotFoundError:
              print('No Bandit report found. Skipping fail step.')
              sys.exit(0)
          highs = [r for r in data.get('results', []) if r.get('issue_severity') in ('HIGH','CRITICAL')]
          if highs:
              print(f"Found {len(highs)} HIGH/CRITICAL issues. Failing build.")
              sys.exit(1)
          print('No HIGH/CRITICAL issues found.')
          EOF
"""

with open(workflow_path, "w") as f:
    f.write(workflow)

print(f"Deliverable 2 written to {workflow_path}")

## 11. Generate Deliverable 3 — Remediation Priority Matrix (CSV)

Creates: `deliverables/remediation_matrix.csv`

In [ ]:
matrix_path = "deliverables/remediation_matrix.csv"

if not top10.empty:
    def remediation_hint(row):
        tid = row.get("test_id", "")
        if tid in ("B105", "B106"):
            return "Remove hardcoded credentials; use env vars or secret manager."
        if tid in ("B301", "B403"):
            return "Replace pickle with safer serialization (e.g., JSON, ONNX)."
        return "Apply secure coding fix based on CWE and test_id."

    df_matrix = top10.copy()
    df_matrix["remediation"] = df_matrix.apply(remediation_hint, axis=1)
    df_matrix["estimated_time_hours"] = 2  # simple default

    df_matrix[[
        "file",
        "line",
        "test_id",
        "cwe",
        "severity",
        "risk_score",
        "remediation",
        "estimated_time_hours",
    ]].to_csv(matrix_path, index=False)

    print(f"Deliverable 3 written to {matrix_path}")
else:
    print("No Bandit findings; remediation matrix not generated.")

## 12. Final confirmation

In [ ]:
print("Deliverables generated:")
print(os.listdir("deliverables"))